In [19]:
import pandas as pd
import numpy as np
from sklearn.metrics import (average_precision_score, roc_auc_score,
                              fbeta_score, classification_report, make_scorer)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier as xgb
from lightgbm import LGBMClassifier as lgb
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.pipeline import Pipeline as SkPipeline


## PIPELINE

El Pipeline convierte age y gender en variables binarias (OneHot), normaliza todas las variables numéricas (StandardScaler) y entrena un modelo de clasificación automáticamente. 

In [6]:
X = pd.read_csv("./data/clean/X_clean.csv")
y = pd.read_csv("./data/clean/y.csv").squeeze()

añadimos categoricas

Vamos a prescindir de las variables merchant y customer.

In [7]:
cat_cols = ["age", "gender", "category"]
num_cols = X.drop(columns=cat_cols).columns.tolist()

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", StandardScaler(), num_cols)
    ]
)

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [10]:
models = {
    "RandomForest_base": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ),
    
    "XGBoost_base": XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42
    ),
    
    "LightGBM_base": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42
    )
}

In [14]:
import time 

def train_base_models(models):
    results = {}

    for name, model in models.items():
        
        clf = Pipeline(steps=[
            ("preprocess", preprocessor),
            ("smote", SMOTE(random_state=42)),
            ("model", model)
        ])
        
        # inicio
        start = time.perf_counter()

        clf.fit(X_train, y_train)

        # fin
        end = time.perf_counter()
        train_time = end - start

        # Predicción
        start_pred = time.perf_counter()

        y_pred_proba = clf.predict_proba(X_test)[:, 1]

        end_pred = time.perf_counter()
        pred_time = end_pred - start_pred
        
        # metrica
        auc = roc_auc_score(y_test, y_pred_proba)
        results[name] = {
            "AUC": auc,
            "Train Time (s)": train_time,
            "Predict Time (s)": pred_time
        }
        
        print(f"{name} → AUC: {auc:.4f} | Train: {train_time:.2f}s | Predict: {pred_time:.4f}s")
    return results

In [17]:
results = train_base_models(models)

results_df = pd.DataFrame.from_dict(results, orient="index")
results_df = results_df.sort_values(by="AUC", ascending=False)

print(results_df)

/Users/martinablay/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


RandomForest_base → AUC: 0.9952 | Train: 76.74s | Predict: 0.2247s


/Users/martinablay/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


XGBoost_base → AUC: 0.9975 | Train: 7.94s | Predict: 0.1565s


/Users/martinablay/Library/Python/3.9/lib/python/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


[LightGBM] [Info] Number of positive: 469954, number of negative: 469954
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.401845 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 72090
[LightGBM] [Info] Number of data points in the train set: 939908, number of used features: 4174
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


/Users/martinablay/Library/Python/3.9/lib/python/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM_base → AUC: 0.9980 | Train: 9.04s | Predict: 0.3562s
                        AUC  Train Time (s)  Predict Time (s)
LightGBM_base      0.997961        9.036546          0.356160
XGBoost_base       0.997510        7.939503          0.156467
RandomForest_base  0.995239       76.736010          0.224732


## AJUSTAR HIPERPARÁMETROS

Vamos a realizar un ajuste de hiperparametros para nuestros modelos, para ello establecemos una función que prueba mediante gridsearch los distintos hiperparametros para nuestros modelos base.

In [21]:
def tune_fraud_models_overnight(X_train, y_train, X_test, y_test,
                                 preprocessor, cv_folds=5, n_jobs=-1):
    """
    Ajuste EXHAUSTIVO de hiperparámetros para RF, LightGBM y XGBoost
    con y sin SMOTE, usando PR-AUC como métrica principal.
    Registra tiempo de entrenamiento de cada combinación.

    ⚠️  Tiempo estimado: 6-18h dependiendo del hardware y tamaño del dataset
    """

    pr_auc_scorer = make_scorer(average_precision_score,
                                needs_proba=True,
                                response_method="predict_proba")
    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)

    neg, pos = np.bincount(y_train)
    scale_pos_weight = neg / pos
    print(f"📊 Ratio desbalance (neg/pos): {scale_pos_weight:.1f}x")

    # ─────────────────────────────────────────────────────────────────
    # GRIDS EXHAUSTIVOS
    # ─────────────────────────────────────────────────────────────────
    models = {

        # ── RANDOM FOREST ─────────────────────────────────────────
        "random_forest": {
            "estimator": RandomForestClassifier(random_state=42, n_jobs=n_jobs),
            "params": {
                "classifier__n_estimators":          [200, 500, 800, 1200],
                "classifier__max_depth":             [10, 20, 30, 40, None],
                "classifier__min_samples_split":     [2, 5, 10, 20],
                "classifier__min_samples_leaf":      [1, 2, 4, 8],
                "classifier__max_features":          ["sqrt", "log2", 0.3, 0.5],
                "classifier__max_samples":           [0.7, 0.8, 0.9, None],
                "classifier__class_weight":          ["balanced", "balanced_subsample", None],
                "classifier__criterion":             ["gini", "entropy"],
                "classifier__min_impurity_decrease": [0.0, 0.001, 0.01],
            }
        },

        # ── LIGHTGBM ──────────────────────────────────────────────
        "lightgbm": {
            "estimator": lgb.LGBMClassifier(random_state=42, verbose=-1,
                                             n_jobs=n_jobs, boosting_type="gbdt"),
            "params": {
                "classifier__n_estimators":       [200, 500, 800, 1200],
                "classifier__max_depth":          [4, 6, 8, 10, -1],
                "classifier__learning_rate":      [0.005, 0.01, 0.05, 0.1, 0.2],
                "classifier__num_leaves":         [31, 63, 127, 255],
                "classifier__min_child_samples":  [5, 10, 20, 50],
                "classifier__subsample":          [0.6, 0.7, 0.8, 1.0],
                "classifier__subsample_freq":     [0, 1, 5],
                "classifier__colsample_bytree":   [0.6, 0.7, 0.8, 1.0],
                "classifier__reg_alpha":          [0, 0.01, 0.1, 0.5, 1.0],
                "classifier__reg_lambda":         [0, 0.01, 0.1, 0.5, 1.0],
                "classifier__min_split_gain":     [0.0, 0.01, 0.1],
                "classifier__class_weight":       ["balanced", None],
                "classifier__scale_pos_weight":   [1, scale_pos_weight],
                "classifier__path_smooth":        [0, 0.1, 1],
            }
        },

        # ── XGBOOST ───────────────────────────────────────────────
        "xgboost": {
            "estimator": xgb.XGBClassifier(
                random_state=42, eval_metric="aucpr",
                verbosity=0, use_label_encoder=False,
                n_jobs=n_jobs, tree_method="hist"
            ),
            "params": {
                "classifier__n_estimators":      [200, 500, 800, 1200],
                "classifier__max_depth":         [3, 4, 6, 8, 10],
                "classifier__learning_rate":     [0.005, 0.01, 0.05, 0.1, 0.2],
                "classifier__subsample":         [0.6, 0.7, 0.8, 1.0],
                "classifier__colsample_bytree":  [0.6, 0.7, 0.8, 1.0],
                "classifier__colsample_bylevel": [0.6, 0.8, 1.0],
                "classifier__colsample_bynode":  [0.6, 0.8, 1.0],
                "classifier__reg_alpha":         [0, 0.01, 0.1, 0.5, 1.0],
                "classifier__reg_lambda":        [0.1, 0.5, 1.0, 5.0],
                "classifier__min_child_weight":  [1, 3, 5, 10, 20],
                "classifier__gamma":             [0, 0.1, 0.5, 1.0, 2.0],
                "classifier__max_delta_step":    [0, 1, 5],
                "classifier__scale_pos_weight":  [1, scale_pos_weight],
                "classifier__grow_policy":       ["depthwise", "lossguide"],
                "classifier__max_bin":           [256, 512],
            }
        },
    }

    smote_params = {
        "smote__k_neighbors":       [3, 5, 7, 10],
        "smote__sampling_strategy": [0.1, 0.25, 0.5, "minority", "auto"],
    }

    results     = {}
    total_start = time.time()

    for model_name, config in models.items():
        n_combos = np.prod([len(v) for v in config["params"].values()])
        n_combos_smote = n_combos * np.prod([len(v) for v in smote_params.values()])

        print(f"\n{'='*60}")
        print(f"  {model_name.upper()}")
        print(f"  Combinaciones sin SMOTE : {n_combos:,}")
        print(f"  Combinaciones con SMOTE : {n_combos_smote:,}")
        print(f"{'='*60}")

        for use_smote in [True, False]:
            tag = f"{model_name}_{'con' if use_smote else 'sin'}_smote"
            print(f"\n  → {'Con' if use_smote else 'Sin'} SMOTE...")

            if use_smote:
                pipeline = ImbPipeline(steps=[
                    ("preprocessor", preprocessor),
                    ("smote",        SMOTE(random_state=42)),
                    ("classifier",   config["estimator"])
                ])
                param_grid = {**config["params"], **smote_params}
            else:
                pipeline = SkPipeline(steps=[
                    ("preprocessor", preprocessor),
                    ("classifier",   config["estimator"])
                ])
                param_grid = config["params"]

            grid_search = GridSearchCV(
                estimator  = pipeline,
                param_grid = param_grid,
                scoring    = pr_auc_scorer,
                cv         = cv,
                n_jobs     = n_jobs,
                verbose    = 2,
                refit      = True,
                error_score= 0.0
            )

            # ── Entrenamiento con cronómetro ──────────────────────
            start_time = time.time()
            grid_search.fit(X_train, y_train)
            elapsed    = time.time() - start_time

            horas    = int(elapsed // 3600)
            minutos  = int((elapsed % 3600) // 60)
            segundos = int(elapsed % 60)
            tiempo_str = f"{horas:02d}h {minutos:02d}m {segundos:02d}s"

            # ── Métricas en test ──────────────────────────────────
            best_pipeline = grid_search.best_estimator_
            y_prob = best_pipeline.predict_proba(X_test)[:, 1]
            y_pred = best_pipeline.predict(X_test)

            auc_test    = roc_auc_score(y_test, y_prob)
            pr_auc_test = average_precision_score(y_test, y_prob)
            f2_test     = fbeta_score(y_test, y_pred, beta=2)
            f1_test     = fbeta_score(y_test, y_pred, beta=1)

            # ── Entrada en results ────────────────────────────────
            results[tag] = {
                "AUC":             auc_test,
                "PR_AUC_test":     pr_auc_test,
                "PR_AUC_cv":       grid_search.best_score_,
                "F2_test":         f2_test,
                "F1_test":         f1_test,
                "SMOTE":           use_smote,
                "model":           model_name,
                "train_time_sec":  round(elapsed, 2),
                "train_time":      tiempo_str,
                "best_params":     grid_search.best_params_,
                "best_estimator":  best_pipeline,
            }

            print(f"\n  ✅ AUC test     : {auc_test:.4f}")
            print(f"  ✅ PR-AUC test  : {pr_auc_test:.4f}")
            print(f"  ✅ PR-AUC cv    : {grid_search.best_score_:.4f}")
            print(f"  ✅ F2 test      : {f2_test:.4f}")
            print(f"  ⏱️  Tiempo       : {tiempo_str}")
            print(f"  📌 Best params  : {grid_search.best_params_}")

    # ── Tiempo total ──────────────────────────────────────────────
    total_elapsed = time.time() - total_start
    th = int(total_elapsed // 3600)
    tm = int((total_elapsed % 3600) // 60)
    ts = int(total_elapsed % 60)
    print(f"\n Tiempo total de ejecución: {th:02d}h {tm:02d}m {ts:02d}s")

    return results


# ─────────────────────────────────────────────────────────────────
# USO + DATAFRAME FINAL
# ─────────────────────────────────────────────────────────────────
results = tune_fraud_models_overnight(
    X_train, y_train, X_test, y_test, preprocessor
)

# ── Tu DataFrame original + columnas nuevas ───────────────────────
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values(by="AUC", ascending=False)

# Separar objetos no tabulares
best_params = results_df.pop("best_params")
best_models = results_df.pop("best_estimator")

# Redondear métricas numéricas
metric_cols = ["AUC", "PR_AUC_test", "PR_AUC_cv", "F2_test", "F1_test"]
results_df[metric_cols] = results_df[metric_cols].astype(float).round(4)

# Tabla final
display_cols = ["model", "SMOTE"] + metric_cols + ["train_time"]
print("\n📊 TABLA FINAL DE RESULTADOS")
print(results_df[display_cols].to_string())

# ── Mejor modelo ──────────────────────────────────────────────────
best_tag   = results_df.index[0]
best_model = best_models[best_tag]

print(f"\n🏆 Mejor modelo  : {best_tag}")
print(f"   Tiempo        : {results_df.loc[best_tag, 'train_time']}")
print(f"   Params        : {best_params[best_tag]}")
print(f"\n{classification_report(y_test, best_model.predict(X_test), target_names=['legítima', 'fraude'])}")


📊 Ratio desbalance (neg/pos): 81.6x


AttributeError: type object 'LGBMClassifier' has no attribute 'LGBMClassifier'

In [37]:
results = tune_fraud_models_overnight(
    X_train, y_train, X_test, y_test, preprocessor
)

# ── Tu DataFrame original + columnas nuevas ───────────────────────
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values(by="AUC", ascending=False)

# Separar objetos no tabulares
best_params = results_df.pop("best_params")
best_models = results_df.pop("best_estimator")

# Redondear métricas numéricas
metric_cols = ["AUC", "PR_AUC_test", "PR_AUC_cv", "F2_test", "F1_test"]
results_df[metric_cols] = results_df[metric_cols].astype(float).round(4)

# Tabla final
display_cols = ["model", "SMOTE"] + metric_cols + ["train_time"]
print("\n📊 TABLA FINAL DE RESULTADOS")
print(results_df[display_cols].to_string())

# ── Mejor modelo ──────────────────────────────────────────────────
best_tag   = results_df.index[0]
best_model = best_models[best_tag]

print(f"\n🏆 Mejor modelo  : {best_tag}")
print(f"   Tiempo        : {results_df.loc[best_tag, 'train_time']}")
print(f"   Params        : {best_params[best_tag]}")
print(f"\n{classification_report(y_test, best_model.predict(X_test), target_names=['legítima', 'fraude'])}")


NameError: name 'make_scorer' is not defined